In [ ]:
import torch
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from ddcc.orientationscore import cakewavelet_stack, wavelet_transform
from ddcc.vesselness import multi_scale_vesselness
from ddcc.algorithms import connected_components, persistence
from pathlib import Path

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern"],
    "font.size": 10.0,
    "text.latex.preamble": r"\usepackage{lmodern} \usepackage{amssymb} \usepackage{amsmath}",
    "image.cmap": "gray",
    "figure.constrained_layout.use": True,
    "xtick.top": False,
    "xtick.bottom": False,
    "ytick.left": False,
    "ytick.right": False,
    "xtick.labeltop": False,
    "xtick.labelbottom": False,
    "ytick.labelleft": False,
    "ytick.labelright": False,
})
latex_line_width = 4.804

device = "cuda"

In [ ]:
# colours
red = "#B6321C" # BrickRed
blue = "#0071BC" # RoyalBlue
green = "#009B55" # ForestGreen
purple = "#613F99" # RoyalPurple

def plot_image(ax, image, **kwargs):
    image = image.cpu()
    return ax.imshow(image, **kwargs)

output_directory = Path("output")
data_directory = Path("data")
config_path = Path("config.json")

In [ ]:
def plot_all_components(components, figsize=(12, 10), fig=None, ax=None):
    N_components = len(components)
    combined = torch.zeros_like(components[0][0])
    for i, component in enumerate(components):
        combined[component.max(0)[0] == 1.] = i + 1
    colors = plt.cm.rainbow(np.linspace(0, 1, N_components))
    colors = np.vstack([[0, 0, 0, 1], colors])
    cmap = mcolors.ListedColormap(colors)
    if fig is None or ax is None:
        fig, ax = plt.subplots(1, 1, figsize=figsize)
    im = ax.imshow(combined.cpu(), cmap=cmap, interpolation='nearest')
    fig.colorbar(im, ticks=np.arange(1, N_components+1), ax=ax)

In [ ]:
# "cross" "line" "circle_cross"
test = "cross"

In [ ]:
dim_K = 32

match test:
    case "cross":
        dim_I, dim_J = 256, 256
        x_min, x_max = -1., 1.
        y_min, y_max = -1., 1.
        x, y = np.linspace(x_min, x_max, dim_I)[:, None], np.linspace(y_min, y_max, dim_J)[None, :]

        u = 1 - (
            ((x - y)**2 < 0.0005) + 
            ((x + y)**2 < 0.0005)
        ) * (np.sin(5. * np.pi * x) >= -0.5)
        u = sp.ndimage.gaussian_filter(u * 1., 1.)
    case "line":
        dim_I, dim_J = 128, 128
        x_min, x_max = -1., 1.
        y_min, y_max = -1., 1.
        x, y = np.linspace(x_min, x_max, dim_I)[:, None], np.linspace(y_min, y_max, dim_J)[None, :]

        u = 1 - (
            (y > -0.05) * (y < 0.05)
        ) * (
            (x < -0.15) + (x > 0.15)
        )
        u = sp.ndimage.gaussian_filter(u * 1., 1.)
    case "circle_cross":
        dim_I, dim_J = 256, 256
        x_min, x_max = -1., 1.
        y_min, y_max = -1., 1.
        x, y = np.linspace(x_min, x_max, dim_I)[:, None], np.linspace(y_min, y_max, dim_J)[None, :]

        u = 1 - ((
            ((x - y)**2 < 0.0005) + 
            ((x + y)**2 < 0.0005)
        ) + (
            (np.sqrt(x**2 + y**2) < 0.51) *
            (np.sqrt(x**2 + y**2) > 0.49)
        ))
        u = sp.ndimage.gaussian_filter(u * 1., 1.)
u = torch.as_tensor(u, device="cuda", dtype=torch.float32)

In [ ]:
cws = cakewavelet_stack(min(dim_I, dim_J), dim_K).to("cuda")
U = wavelet_transform(u, cws)

In [ ]:
K = 0
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
cbar = plot_image(ax[0], cws[K])
fig.colorbar(cbar, ax=ax[0])
cbar = plot_image(ax[1], U[K])
fig.colorbar(cbar, ax=ax[1]);

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
cbar = plot_image(ax[0], u)
fig.colorbar(cbar, ax=ax[0])
cbar = plot_image(ax[1], torch.clip(U.sum(0), 0., 1.))
fig.colorbar(cbar, ax=ax[1]);

In [ ]:
σs_s_list = (1., 2.)
σ_0 = 0.5
σ_s_ext = 0.5
σ_o_ext = 0.5
V = multi_scale_vesselness(U, σs_s_list, σ_0, σ_s_ext, σ_o_ext)
λ = 100
p = 2
C = 1 / (1 + λ * V**p)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
cbar = plot_image(ax[0], u)
fig.colorbar(cbar, ax=ax[0])
cbar = plot_image(ax[1], C.min(0)[0])
fig.colorbar(cbar, ax=ax[1]);

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 10))
cbar = plot_image(ax[0, 0], C[0])
fig.colorbar(cbar, ax=ax[0, 0])
cbar = plot_image(ax[0, 1], C[4])
fig.colorbar(cbar, ax=ax[0, 1])
cbar = plot_image(ax[1, 0], C[8])
fig.colorbar(cbar, ax=ax[1, 0])
cbar = plot_image(ax[1, 1], C[12])
fig.colorbar(cbar, ax=ax[1, 1]);

In [ ]:
on_data = (C < 0.1) * 1.

In [ ]:
offset = 0
fig, ax = plt.subplots(2, 2, figsize=(12, 10))
cbar = plot_image(ax[0, 0], on_data[0+offset])
fig.colorbar(cbar, ax=ax[0, 0])
cbar = plot_image(ax[0, 1], on_data[4+offset])
fig.colorbar(cbar, ax=ax[0, 1])
cbar = plot_image(ax[1, 0], on_data[8+offset])
fig.colorbar(cbar, ax=ax[1, 0])
cbar = plot_image(ax[1, 1], on_data[12+offset])
fig.colorbar(cbar, ax=ax[1, 1]);

In [ ]:
components = connected_components(on_data, C, 100., (1., 0., 0.00025), threshold=0.8)

In [ ]:
distance_map, persistence_diagram = persistence(on_data, C, torch.linspace(0, 100, 10), (1., 0., 0.00025), n_max=6000)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
plot_all_components(components, fig=fig, ax=ax)

In [ ]:
fig, ax = plt.subplots(1, 1)
cbar = plot_image(ax, distance_map.amin(0) < 100.)
fig.colorbar(cbar, ax=ax);

In [ ]:
plt.plot(persistence_diagram)